In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/pet-breed-classifier

!pip install -q gradio onnxruntime fastapi uvicorn pillow
print('მზადაა')

Mounted at /content/drive
/content/drive/MyDrive/pet-breed-classifier
მზადაა


In [1]:
%%writefile app.py
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
from PIL import Image

ROOT = Path(__file__).parent
MODEL_PATH = ROOT / "models" / "resnet18.onnx"
CLASSES_PATH = ROOT / "models" / "classes.json"

# იგივე ნორმალიზაცია, რაც ვარჯიშზე გამოვიყენეთ
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

classes = json.load(open(CLASSES_PATH, encoding="utf-8"))
session = ort.InferenceSession(str(MODEL_PATH), providers=["CPUExecutionProvider"])


def preprocess(img):
    img = img.convert("RGB").resize((224, 224), Image.BILINEAR)
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1)[None]  # [1, 3, 224, 224]


def predict(img, k=3):
    logits = session.run(None, {"input": preprocess(img)})[0][0]
    e = np.exp(logits - logits.max())
    probs = e / e.sum()
    top = probs.argsort()[::-1][:k]
    return {classes[i]: float(probs[i]) for i in top}

Writing app.py


In [2]:
%%writefile -a app.py


import io

import gradio as gr
from fastapi import FastAPI, File, UploadFile

# --- API ---
api = FastAPI(title="Pet Breed Classifier")


@api.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    img = Image.open(io.BytesIO(await file.read()))
    return {"top3": predict(img)}


# --- ვებ-ინტერფეისი ---
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="ატვირთე კატის ან ძაღლის სურათი"),
    outputs=gr.Label(num_top_classes=3, label="Top-3 პროგნოზი"),
    title="Oxford Pet-ის ჯიშების ამომცნობი (37 ჯიში)",
    description="ResNet18 (fine-tuned), ONNX Runtime-ზე. Test accuracy 89.1%.",
)

# ორივე ერთ აპლიკაციაში: "/" არის ინტერფეისი, "/predict" არის API
app = gr.mount_gradio_app(api, demo, path="/")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=7860)

Appending to app.py


In [6]:
import os
if os.path.exists('final_models.zip'):
    !unzip -o -q final_models.zip
    !ls -lh models
else:
    print('final_models.zip ჯერ არ ჩანს Drive-ში, ატვირთვა დაასრულე და ისევ გაუშვი')

total 118M
-rw------- 1 root root 1.6M Oct 27 12:06 baseline_cnn.pth
-rw------- 1 root root  580 Oct 28 09:46 classes.json
-rw------- 1 root root 6.6M Oct 28 08:08 medium_cnn_30ep.pth
-rw------- 1 root root 6.6M Oct 28 07:23 medium_cnn.pth
-rw------- 1 root root 6.6M Oct 27 16:38 mediumcnn.pth
-rw------- 1 root root  43M Oct 28 09:32 resnet18_finetune.pth
-rw------- 1 root root  11M Oct 28 09:43 resnet18_int8.onnx
-rw------- 1 root root  43M Oct 28 09:42 resnet18.onnx


In [5]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/pet-breed-classifier

!unzip -o -q final_models.zip
!ls -lh models

Mounted at /content/drive
/content/drive/MyDrive/pet-breed-classifier
total 118M
-rw------- 1 root root 1.6M Oct 27 12:06 baseline_cnn.pth
-rw------- 1 root root  580 Oct 28 09:46 classes.json
-rw------- 1 root root 6.6M Oct 28 08:08 medium_cnn_30ep.pth
-rw------- 1 root root 6.6M Oct 28 07:23 medium_cnn.pth
-rw------- 1 root root 6.6M Oct 27 16:38 mediumcnn.pth
-rw------- 1 root root  43M Oct 28 09:32 resnet18_finetune.pth
-rw------- 1 root root  11M Oct 28 09:43 resnet18_int8.onnx
-rw------- 1 root root  43M Oct 28 09:42 resnet18.onnx


In [2]:
import glob, random
from PIL import Image
from app import predict

path = random.choice(glob.glob('data/oxford-iiit-pet/images/*.jpg'))
print('ფაილი:', path)

img = Image.open(path)
for breed, p in predict(img).items():
    print(f'  {breed:28s} {p:.1%}')

ფაილი: data/oxford-iiit-pet/images/beagle_46.jpg
  Beagle                       95.1%
  Basset Hound                 4.2%
  Shiba Inu                    0.2%


In [6]:
%%writefile -a /content/drive/MyDrive/pet-breed-classifier/app.py


import io

import gradio as gr
from fastapi import FastAPI, File, UploadFile

# --- API ---
api = FastAPI(title="Pet Breed Classifier")


@api.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    img = Image.open(io.BytesIO(await file.read()))
    return {"top3": predict(img)}


# --- ვებ-ინტერფეისი ---
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="ატვირთე კატის ან ძაღლის სურათი"),
    outputs=gr.Label(num_top_classes=3, label="Top-3 პროგნოზი"),
    title="Oxford Pet-ის ჯიშების ამომცნობი (37 ჯიში)",
    description="ResNet18 (fine-tuned), ONNX Runtime-ზე. Test accuracy 89.1%.",
)

# ორივე ერთ აპლიკაციაში: "/" არის ინტერფეისი, "/predict" არის API
app = gr.mount_gradio_app(api, demo, path="/")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=7860)

Appending to /content/drive/MyDrive/pet-breed-classifier/app.py


In [7]:
!grep -c "" /content/drive/MyDrive/pet-breed-classifier/app.py
!grep -n "^demo\|^api\|^app " /content/drive/MyDrive/pet-breed-classifier/app.py

64
41:api = FastAPI(title="Pet Breed Classifier")
51:demo = gr.Interface(
60:app = gr.mount_gradio_app(api, demo, path="/")


In [8]:
import sys, importlib
sys.path.insert(0, '/content/drive/MyDrive/pet-breed-classifier')
import app
importlib.reload(app)

print('demo:', hasattr(app, 'demo'), '| api:', hasattr(app, 'api'))
app.demo.launch(share=True, debug=False)

demo: True | api: True
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d4768768427f9bde0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
